In [2]:
from vllm import LLM, SamplingParams

from env_vars import CACHE_DIR

In [ ]:
llm = LLM(model='Qwen/Qwen3-4B', download_dir=CACHE_DIR, max_model_len=16384)

INFO 04-07 14:06:30 [utils.py:233] non-default args: {'download_dir': '/fs/nexus-scratch/snawathe/hf_home/', 'max_model_len': 16384, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B'}
INFO 04-07 14:06:30 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-07 14:06:30 [model.py:1678] Using max model len 16384
INFO 04-07 14:06:30 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-07 14:06:30 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=2277001) INFO 04-07 14:06:34 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=16384, download_dir='/fs/nexus-scratch/snawathe/hf_home/', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_s

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=2277001) INFO 04-07 14:06:40 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/3)
(EngineCore pid=2277001) INFO 04-07 14:06:41 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/3)
(EngineCore pid=2277001) INFO 04-07 14:06:41 [weight_utils.py:825] Prefetching checkpoint files: 30% (3/3)
(EngineCore pid=2277001) INFO 04-07 14:06:41 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.64s
(EngineCore pid=2277001) INFO 04-07 14:06:42 [default_loader.py:384] Loading weights took 1.23 seconds
(EngineCore pid=2277001) INFO 04-07 14:06:42 [gpu_model_runner.py:4820] Model loading took 7.56 GiB memory and 4.031639 seconds
(EngineCore pid=2277001) INFO 04-07 14:06:50 [backends.py:1051] Using cache directory: /nfshomes/snawathe/.cache/vllm/torch_compile_cache/914b9cf453/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2277001) INFO 04-07 14:06:50 [backends.py:1111] Dynamo bytecode transform time: 7.29 s
(EngineCore pid=2277001

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|█████████████████████████████████████████████████████████████████████████████████| 51/51 [00:02<00:00, 20.03it/s]
Capturing CUDA graphs (decode, FULL): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:01<00:00, 25.34it/s]


(EngineCore pid=2277001) INFO 04-07 14:07:00 [gpu_model_runner.py:6046] Graph capturing finished in 5 secs, took 0.52 GiB
(EngineCore pid=2277001) INFO 04-07 14:07:00 [gpu_worker.py:597] CUDA graph pool memory: 0.52 GiB (actual), 0.51 GiB (estimated), difference: 0.0 GiB (0.8%).
(EngineCore pid=2277001) INFO 04-07 14:07:01 [core.py:283] init engine (profile, create kv cache, warmup model) took 18.25 seconds


(EngineCore pid=2277001) INFO 04-07 14:25:19 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=2277001) INFO 04-07 14:25:19 [core.py:1233] Shutdown complete


In [4]:
sampling_params = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=16384, n=8)

In [5]:
# question = 'Every morning Aya goes for a $9$-kilometer-long walk and stops at a coffee shop afterwards. When she walks at a constant speed of $s$ kilometers per hour, the walk takes her 4 hours, including $t$ minutes spent in the coffee shop. When she walks $s+2$ kilometers per hour, the walk takes her 2 hours and 24 minutes, including $t$ minutes spent in the coffee shop. Suppose Aya walks at $s+\\frac{1}{2}$ kilometers per hour. Find the number of minutes the walk takes her, including the $t$ minutes spent in the coffee shop.'
question = 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$'
messages = [
	{'role': 'user', 'content': 'Reason step by step, and place your final answer ins \\boxed{}. Question: ' + question}
]

In [6]:
outputs = llm.chat([messages], sampling_params, chat_template_kwargs={'enable_thinking': True})

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 04-07 14:07:02 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|                                                                             | 0/8 [00…

In [7]:
outputs[0].prompt

'<|im_start|>user\nReason step by step, and place your final answer ins \\boxed{}. Question: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$<|im_end|>\n<|im_start|>assistant\n'

In [16]:
from vllm import RequestOutput, CompletionOutput

In [ ]:
def serialize_request_output(output: RequestOutput) -> dict:
	return {
	'request_id': output.request_id,
	'prompt': output.prompt,
	'prompt_token_ids': output.prompt_token_ids,
	'prompt_logprobs': output.prompt_logprobs,
	'outputs': output.outputs,
	'finished': output.finished,
}

In [10]:
RequestOutput(**{
	'request_id': outputs[0].request_id,
	'prompt': outputs[0].prompt,
	'prompt_token_ids': outputs[0].prompt_token_ids,
	'prompt_logprobs': outputs[0].prompt_logprobs,
	'outputs': outputs[0].outputs,
	'finished': outputs[0].finished,
})

RequestOutput(request_id=0, prompt='<|im_start|>user\nReason step by step, and place your final answer ins \\boxed{}. Question: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$<|im_end|>\n<|im_start|>assistant\n', prompt_token_ids=[151644, 872, 198, 25139, 3019, 553, 3019, 11, 323, 1992, 697, 1590, 4226, 1640, 1124, 79075, 46391, 15846, 25, 7169, 279, 1459, 4930, 15, 11, 18, 15087, 304, 51424, 13934, 311, 24660, 13934, 13, 220, 11252, 697, 4226, 304, 279, 1352, 4930, 81, 26266, 15976, 98406, 1380, 400, 81, 861, 220, 15, 3, 323, 400, 15, 1124, 273, 1124, 15976, 366, 220, 17, 1124, 2493, 2418, 151645, 198, 151644, 77091, 198], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text="<think>\nOkay, so I need to convert the point (0, 3) from rectangular coordinates to polar coordinates. Hmm, let me remember how to do 

In [ ]:
def serialize_completion_output(output: CompletionOutput) -> dict:
	return {
		'index': output.index,
		'text': output.text,
		'token_ids': output.token_ids,
		'logprobs': output.logprobs,
		'cumulative_logprob': output.cumulative_logprob,
	}

In [21]:
CompletionOutput(**{
	'index': outputs[0].outputs[0].index,
	'text': outputs[0].outputs[0].text,
	'token_ids': outputs[0].outputs[0].token_ids,
	'logprobs': outputs[0].outputs[0].logprobs,
	'cumulative_logprob': outputs[0].outputs[0].cumulative_logprob,
})

CompletionOutput(index=0, text="<think>\nOkay, so I need to convert the point (0, 3) from rectangular coordinates to polar coordinates. Hmm, let me remember how to do this. I think polar coordinates are given by (r, θ), where r is the distance from the origin to the point, and θ is the angle made with the positive x-axis. \n\nFirst, I need to find r. The formula for r in polar coordinates is the square root of (x squared plus y squared), right? So, since the point is (0, 3), x is 0 and y is 3. Let me write that down:\n\nr = √(x² + y²) = √(0² + 3²) = √(0 + 9) = √9 = 3. \n\nOkay, so r is 3. That seems straightforward. Now, I need to find θ. The formula for θ is the arctangent of y over x, so θ = arctan(y/x). But wait, since x is 0 here, that would be arctan(3/0). Hmm, arctan of infinity? Because dividing by zero is undefined, but approaching infinity. \n\nI remember that arctan(∞) is π/2, right? Because the tangent of π/2 is undefined, but approaching from the left, it goes to positive i

In [ ]:
def deserialize_completion_output(data: dict) -> CompletionOutput:
	return CompletionOutput(**data)

def deserialize_request_output(data: dict) -> RequestOutput:
	data['outputs'] = list(map(deserialize_completion_output, data['outputs']))
	return RequestOutput(**data)

In [15]:
outputs[0].outputs[0]

CompletionOutput(index=0, text="<think>\nOkay, so I need to convert the point (0, 3) from rectangular coordinates to polar coordinates. Hmm, let me remember how to do this. I think polar coordinates are given by (r, θ), where r is the distance from the origin to the point, and θ is the angle made with the positive x-axis. \n\nFirst, I need to find r. The formula for r in polar coordinates is the square root of (x squared plus y squared), right? So, since the point is (0, 3), x is 0 and y is 3. Let me write that down:\n\nr = √(x² + y²) = √(0² + 3²) = √(0 + 9) = √9 = 3. \n\nOkay, so r is 3. That seems straightforward. Now, I need to find θ. The formula for θ is the arctangent of y over x, so θ = arctan(y/x). But wait, since x is 0 here, that would be arctan(3/0). Hmm, arctan of infinity? Because dividing by zero is undefined, but approaching infinity. \n\nI remember that arctan(∞) is π/2, right? Because the tangent of π/2 is undefined, but approaching from the left, it goes to positive i

In [ ]:
500 * 256 * 50/8 /60 /60 / 40